# Pré-projeto: objetivo, perguntas e modelagem

Este notebook registra o planejamento do projeto antes da construção da camada ouro: o problema, as perguntas, os dados disponíveis, a arquitetura do pipeline e o raciocínio que levou ao modelo das tabelas finais.


## 1. Objetivo

Entender os hábitos de consumo e de venda no marketplace Olist entre 2017 e 2018: onde estão os compradores e os vendedores, quanto se gasta e se vende em cada estado, que categorias concentram a demanda, como os clientes pagam e como avaliam suas compras ao longo do tempo.

O projeto observa o marketplace pelas duas pontas da transação. Do lado do **comprador**, interessa o pedido, ou seja, quanto foi pago, de que forma, com que satisfação. Do lado do **vendedor**, interessa o item vendido: o que foi vendido, por quanto e de onde saiu. Isso ocorre pelo pedido poder conter itens de diferentes vendedores. Essa distinção orienta toda a modelagem.

## 2. Perguntas

1. Qual estado tem o maior número de compras online?
2. Qual estado tem o maior gasto com compras online?
3. Qual estado mais vende em número de vendas?
4. Qual estado mais vende em valor?
5. Quais são as notas dadas pelos clientes, no geral e por estado?
6. Qual o método de pagamento mais utilizado?
7. Qual categoria apresenta mais demanda?
8. Houve mudança na média das avaliações ao longo dos trimestres?
9. Qual o crescimento entre o primeiro semestre de 2017 e o primeiro semestre de 2018?
10. Qual a proporção de compras pagas à vista e de forma parcelada?
11. Qual o valor médio das compras à vista e das parceladas?


## 3. Dados

**Fonte:** *Brazilian E-Commerce Public Dataset by Olist*, publicado no Kaggle (`olistbr/brazilian-ecommerce`). Pedidos reais realizados na plataforma entre 2016 e 2018, anonimizados.

**Estrutura:** nove tabelas relacionais, normalizadas, pensadas para registrar a operação do marketplace.

| Tabela | Conteúdo | Chave |
|---|---|---|
| `olist_orders_dataset` | Pedidos, status e datas de cada etapa | `order_id` |
| `olist_customers_dataset` | Cliente de cada pedido, com cidade, UF e CEP | `customer_id` |
| `olist_order_items_dataset` | Itens de cada pedido, com preço e frete | `order_id` + `order_item_id` |
| `olist_order_payments_dataset` | Pagamentos de cada pedido, com meio, parcelas e valor | `order_id` + `payment_sequential` |
| `olist_order_reviews_dataset` | Avaliações dos pedidos, com nota e comentário | `review_id` + `order_id` |
| `olist_products_dataset` | Produtos, com categoria, medidas e peso | `product_id` |
| `olist_sellers_dataset` | Vendedores, com cidade, UF e CEP | `seller_id` |
| `olist_geolocation_dataset` | Coordenadas por prefixo de CEP | sem chave |
| `product_category_name_translation` | Tradução das categorias para o inglês | `product_category_name` |


## 4. Arquitetura do pipeline

O pipeline segue a arquitetura medalhão, no catálogo `pospucrio`, com um esquema por camada.

| Camada | Esquema | Conteúdo | Notebook |
|---|---|---|---|
| Bronze | `pospucrio.bronze` | Tabelas exatamente como vieram da fonte, todas as colunas como texto. Arquivos CSV originais guardados no volume `raw`. | `Bronze_Ingestao` |
| Prata | `pospucrio.silver` | As mesmas tabelas, com tipagem correta, padronização e correções. | `Silver_Limpeza` |
| Ouro | `pospucrio.gold` | Tabelas modeladas para responder às perguntas. | `Gold_Modelagem` |

O diagnóstico de qualidade fica no notebook `Analises_Bronze-Prata`, fora da sequência do pipeline: ele investiga e justifica, mas não grava tabelas. A análise final fica num notebook próprio, que apenas lê a camada ouro.

O download do Kaggle é salvo no disco local da máquina que executa o notebook, que o Spark não enxerga. O volume é armazenamento em nuvem acessível ao Spark, e ainda preserva os arquivos originais para reprocessamento.


## 5. Método de modelagem

A camada ouro foi desenhada a partir das perguntas, e não da estrutura da origem. Para cada pergunta foram definidos:

1. **Métrica:** o que se conta ou soma.
2. **Agrupamento:** por o quê se agrupa.
3. **Filtro:** quais linhas entram.
4. **Granularidade:** em que nível a conta precisa ser feita.

Agrupando as perguntas pela granularidade, as tabelas aparecem. As colunas de cada tabela são tudo que as perguntas daquele grupo precisam para agrupar, somar e filtrar.

| # | Pergunta | Métrica | Agrupa por | Filtro | Granularidade |
|---|---|---|---|---|---|
| 1 | Estado com mais compras | Contagem de pedidos | Estado do cliente | Entregues | Pedido |
| 2 | Estado com maior gasto | Soma do valor pago | Estado do cliente | Entregues | Pedido |
| 3 | Estado que mais vende (número) | Contagem de itens | Estado do vendedor | Entregues | Item |
| 4 | Estado que mais vende (valor) | Soma do preço | Estado do vendedor | Entregues | Item |
| 5 | Notas | Distribuição, média e % de notas 1 e 2 | Estado do cliente | Todos | Pedido |
| 6 | Método mais usado | Contagem de pedidos | Método principal | Entregues | Pedido |
| 7 | Categoria de maior demanda | Contagem de itens | Categoria | Entregues | Item |
| 8 | Evolução das notas | Média e % de notas 1 e 2 | Trimestre | Todos | Pedido |
| 9 | Crescimento entre semestres | Pedidos e valor pago | Semestre | Entregues | Pedido |
| 10 | Proporção à vista e parcelado | Contagem de pedidos | Forma de pagamento | Entregues | Pedido |
| 11 | Valor médio à vista e parcelado | Média e mediana do valor pago | Forma de pagamento | Entregues | Pedido |

Duas granularidades resultam: **pedido**, com nove perguntas, e **item**, com três.

## 6. O princípio central: compra é pedido, venda é item

O consumidor compra um pedido, que pode conter vários produtos de vários vendedores, localizados em estados diferentes. O vendedor vende itens. Por isso:

- Perguntas sobre o comprador (quantas compras, quanto gastou, como pagou, que nota deu) são respondidas no nível do **pedido**.
- Perguntas sobre o vendedor (quanto vendeu, de onde, que categoria) são respondidas no nível do **item**.

Duas perguntas parecidas mostram a diferença: "estado com mais compras" usa o estado do cliente, e cada pedido tem um. Já "estado que mais vende" usa o estado do vendedor, e um mesmo pedido pode ter vendedores de estados diferentes. O estado do vendedor não é atributo do pedido, só do item. O mesmo vale para a categoria, um pedido com uma cafeteira e um livro não tem uma categoria.

## 7. Modelo da camada ouro


### 7.1 `gold.pedidos`: a perspectiva do comprador

Uma linha por pedido.

| Coluna | Descrição | Origem |
|---|---|---|
| `order_id` | Identificador do pedido (chave) | orders |
| `data_compra` | Data e hora da compra | orders |
| `ano`, `semestre`, `trimestre` | Derivadas da data de compra | orders |
| `estado_cliente` | UF do cliente | customers |
| `status` | Status do pedido | orders |
| `valor_pago` | Soma de todos os pagamentos do pedido | order_payments |
| `forma_pagamento` | À vista ou parcelado | order_payments |
| `metodo_principal` | Meio de pagamento principal | order_payments |
| `nota` | Nota da avaliação do pedido | order_reviews |

### 7.2 `gold.itens`: a perspectiva do vendedor

Uma linha por item de pedido.

| Coluna | Descrição | Origem |
|---|---|---|
| `order_id`, `order_item_id` | Chave composta do item | order_items |
| `data_compra` | Data e hora da compra do pedido | orders |
| `ano`, `semestre`, `trimestre` | Derivadas da data de compra | orders |
| `estado_cliente` | UF de quem comprou | customers |
| `estado_vendedor` | UF de quem vendeu | sellers |
| `categoria` | Categoria do produto, em português | products |
| `preco` | Preço do item, sem frete | order_items |
| `status` | Status do pedido | orders |

Cada linha de `gold.itens` é uma unidade vendida: a quantidade vendida é a contagem de linhas.

## 8. Regras de negócio

**Compra efetivada.** Contam como compra apenas pedidos com status `delivered`. O diagnóstico mostrou que pedidos em status intermediário (shipped, processing, invoiced) aparecem em volume estável desde o início de 2017: não estão em curso, pararam e nunca foram concluídos.

**Valor do comprador.** O `valor_pago` é a soma de todos os pagamentos do pedido: produtos, frete, juros de parcelamento e vouchers. Mede o que o comprador pagou.

**Valor do vendedor.** O `preco` é o preço do item, sem frete: o valor correspondente à venda do produto.

**Voucher.** Entra no valor pago. O diagnóstico mostrou que, em 98.362 pedidos, a soma dos pagamentos com voucher incluído bate com preço mais frete dos itens, ou seja, o voucher cobre parte do valor da compra, como um meio de pagamento. Além disso, a origem do voucher é desconhecida, podendo ser cupom, mas também crédito de devolução ou vale-presente.

**Forma de pagamento.** O pedido é parcelado quando algum pagamento, excluídos os vouchers, tem mais de uma parcela, caso contrário, é à vista. O critério usa o **maior** número de parcelas, e não a soma, ou seja, dois cartões pagos em uma parcela cada são uma compra considerada à vista nessa analise. Pedidos pagos só com voucher são à vista. Pedidos cujo pagamento tem o número de parcelas nulo (registro inválido na origem) ficam com forma de pagamento nula e saem apenas das perguntas de parcelamento.

**Método principal.**
- Quando o pedido combina voucher com outro meio, o método principal é o outro meio.
- Quando o pedido é pago só com voucher, o método principal é o voucher.
- Quando combina dois meios que não são voucher, prevalece o de maior valor pago.
- Pedidos com pagamento `not_defined` ficam com método nulo.

**Nota.** A avaliação está vinculada ao pedido. Quando um pedido tem mais de uma, prevalece a de resposta mais recente (`review_answer_timestamp`). Em caoso de empate, a de criação mais recente, persistindo, o maior `review_id`.

**Status mantido na tabela.** Todos os pedidos ficam em `gold.pedidos`, e o filtro de entregues é aplicado em cada consulta. As perguntas de compra filtram entregues, mas as de avaliação usam todos.

**Categoria.** Nomes originais, em português, com o sublinhado trocado por espaço. Produtos sem categoria recebem o rótulo "sem categoria".

**Período.**
- As tabelas incluem os pedidos a partir de janeiro de 2017.
- Evolução trimestral: do 1º trimestre de 2017 ao 2º de 2018, os seis trimestres completos.
- Comparações entre anos: janeiro a agosto de 2017 contra janeiro a agosto de 2018.
- Crescimento semestral: primeiro semestre de 2017 contra primeiro semestre de 2018, completos nos dois anos.

## 9. Discussões e alternativas consideradas

**Uma tabela ou duas?** A primeira ideia foi uma tabela de compras e outra de vendas, com os mesmos dados e só o estado trocado. Seriam a mesma tabela com duas colunas de estado. O que de fato obriga a separar é a granularidade. A distinção entre pedido e item vendido foi fundamental para criar essa modelagem.

**Pagamento dentro de pedidos ou numa tabela própria?** O pagamento tem granularidade própria (um pedido pode ter vários). A alternativa de uma terceira tabela preservaria todos os meios usados. Optei por resumir no pedido, com a regra do método principal, porque as perguntas tratam do pagamento como atributo da compra. A perda de informação é pequena: 2.246 pedidos combinam meios, quase todos cartão com voucher.

**Esquema estrela ou tabelas largas?** O esquema estrela compensa quando as dimensões são ricas e compartilhadas, e quando as perguntas não são conhecidas de antemão. Aqui, as perguntas usam essencialmente o estado do cliente e do vendedor e a categoria do produto, dimensões com um atributo útil acrescentariam joins sem ganho. O volume é pequeno e o ouro é regenerado a partir da prata a cada execução, então a redundância das tabelas largas não pesa. O modelo corresponde ao **modelo flat** descrito no enunciado. Se o projeto crescesse para perguntas sobre perfil de cliente, atributos de produto ou logística, o esquema estrela passaria a fazer sentido.

**Filtrar entregues na tabela ou na consulta?** Filtrar na tabela removeria as avaliações de quem não recebeu o produto, justamente as piores. A média de notas sairia artificialmente alta. Por isso o filtro fica na consulta.

**Voucher: desconto ou pagamento?** A primeira leitura foi de desconto, que o cliente não desembolsa. Contra ela, o dado não informa a origem do voucher, e a conferência de valores mostrou que ele integra o total pago. Decidiu-se incluí-lo no valor e excluí-lo apenas da classificação de parcelamento, onde não representa uma decisão de parcelar.

**Parcelas: soma ou máximo?** Somar parcelas de pagamentos diferentes classificaria dois cartões à vista como uma compra em duas parcelas. O máximo representa a decisão de parcelar.

**Duas avaliações para o mesmo pedido: manter ou escolher?** Manter as duas quebraria a granularidade da tabela de pedidos e duplicaria o valor pago. A primeira regra escolhia pela data de criação, mas ela só registra o dia (157 empates) e corresponde ao envio da pesquisa, não à resposta. A data de resposta, com hora completa, representa melhor a avaliação mais recente do cliente.

**Incluir o fim de 2016?** O último trimestre de 2016 completaria dois anos de série, mas tem 324 pedidos em outubro, nenhum em novembro e 1 em dezembro. Comparar essa média com trimestres de dezenas de milhares de pedidos faria oscilações parecerem tendência.

**Cortar o fim de 2017 para comparar com 2018?** Os dados de 2018 terminam em agosto. Comparar 2017 inteiro com 2018 incluiria só de um lado a Black Friday e o Natal. Mas o corte vale apenas para comparações entre anos. Na série trimestral de notas, o quarto trimestre de 2017 é mantido, a sazonalidade é parte do que se quer observar, e o pico de novembro pode explicar variações na satisfação.

**Traduzir as categorias?** A tabela de tradução leva as categorias para o inglês. Como a análise é em português, os nomes originais são mais naturais, e a decisão dispensa tratar as duas categorias sem tradução.

**Média ou mediana?** Preços e valores pagos têm cauda longa: poucos itens muito caros puxam a média para cima. As notas são polarizadas: a nota 1 é mais frequente que a 2 e a 3 somadas. Por isso, valores médios são reportados com a mediana ao lado, e notas médias com a proporção de notas 1 e 2.